# 4.4 · Hooks y persistencia

**Problema 1:** no sabemos qué ha hecho el agente ni por qué. **Problema 2:** queremos poder retomar una conversación más tarde.

**Al terminar tendrás:** un hook de auditoría que registra todas las decisiones en `logs/auditoria.jsonl`, un hook que bloquea una acción, y una sesión reanudada después de reiniciar el kernel.

Un agente que no puedes observar es un agente que no puedes depurar ni poner en producción.

## Preparación

Todo lo anterior viene de `taller_utils.py`. `BASE` son las opciones comunes del agente, como diccionario: así cada sección crea sus `ClaudeAgentOptions(**BASE, ...)` añadiendo solo lo que cambia.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import herramientas
from taller_utils import (
    imprimir, buscar_incidencias, crear_incidencia, cerrar_incidencia,
    CLASIFICADOR, HERRAMIENTAS_KB,
)
from claude_agent_sdk import query, create_sdk_mcp_server, ClaudeAgentOptions, HookMatcher, ResultMessage

soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[buscar_incidencias, crear_incidencia, cerrar_incidencia])
SYSTEM = ("Eres un agente de soporte de nivel 1. Respondes en español y de forma breve. "
          "Antes de crear una incidencia, comprueba que no existe ya una abierta sobre el mismo problema. "
          "La prioridad de cada incidencia se asigna según los criterios oficiales de prioridad: no la supongas.")
BASE = dict(
    model=MODEL,
    system_prompt=SYSTEM,
    mcp_servers={"soporte": soporte},
    tools=["Task"],
    allowed_tools=["Task", "mcp__soporte__buscar_incidencias", "mcp__soporte__crear_incidencia",
                   "mcp__soporte__cerrar_incidencia", *HERRAMIENTAS_KB],
    agents={"clasificador": CLASIFICADOR},   # con su propio servidor de la KB
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},
    max_turns=15,
    setting_sources=[],
)

In [ ]:
CORREO = "Llega este correo de Initech: 'nadie de mi equipo puede entrar desde esta mañana'. Regístralo."

## Hook de auditoría

Un *hook* es una función que el SDK llama en momentos concretos del bucle. Los dos que nos interesan:

- `PreToolUse`: justo **antes** de ejecutar una herramienta (con lo que el modelo ha pedido).
- `PostToolUse`: justo **después** (con lo que ha devuelto).

Escribimos cada evento como una línea JSON en `logs/auditoria.jsonl`. La ruta, absoluta.

In [ ]:
import json, datetime

LOG_AUDITORIA = Path("../../logs/auditoria.jsonl").resolve()
LOG_AUDITORIA.unlink(missing_ok=True)   # empezamos con el log vacío

async def auditar(input_data, tool_use_id, context):
    registro = {
        "ts": datetime.datetime.now().isoformat(),
        "evento": input_data["hook_event_name"],
        "herramienta": input_data.get("tool_name"),
        "entrada": input_data.get("tool_input"),
        "tool_use_id": tool_use_id,
    }
    if input_data["hook_event_name"] == "PostToolUse":
        registro["salida"] = str(input_data.get("tool_response"))[:500]
    with open(LOG_AUDITORIA, "a") as f:
        f.write(json.dumps(registro, ensure_ascii=False) + "\n")
    return {}

Un hook que devuelve `{}` no interfiere: **solo observa**. `HookMatcher` sin `matcher` se aplica a todas las herramientas.

También guardamos el `session_id` del `ResultMessage`: lo usaremos al final del notebook.

In [ ]:
def guardar_sesion(session_id):
    Path("../../logs/ultima_sesion.txt").write_text(session_id)

options = ClaudeAgentOptions(
    **BASE,
    hooks={
        "PreToolUse":  [HookMatcher(hooks=[auditar])],
        "PostToolUse": [HookMatcher(hooks=[auditar])],
    },
)

herramientas.restaurar_datos()
async for msg in query(prompt=CORREO, options=options):
    imprimir(msg)
    if isinstance(msg, ResultMessage):
        guardar_sesion(msg.session_id)

El log, como tabla:

In [ ]:
import pandas as pd
pd.read_json(LOG_AUDITORIA, lines=True)[["ts", "evento", "herramienta", "entrada"]]

Se ve en orden la búsqueda, la delegación al clasificador (`Agent`), cada lectura de la base de conocimiento del subagente y la creación final. Mucho más legible que el `.jsonl` en crudo, y la misma celda servirá para depurar en el bloque 5.

## Hook que bloquea

Un hook también puede **decidir**. Política: las incidencias de prioridad alta solo las cierra una persona.

`matcher` limita el hook a una herramienta. Si devuelve `permissionDecision: "deny"`, la herramienta no se ejecuta y el modelo recibe el motivo.

In [ ]:
async def proteger_cierres(input_data, tool_use_id, context):
    inc = herramientas.obtener(input_data["tool_input"]["id"])
    if inc and inc["prioridad"] == "alta":
        return {"hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": "Las incidencias de prioridad alta solo las cierra un humano.",
        }}
    return {}

options = ClaudeAgentOptions(
    **BASE,
    hooks={
        "PreToolUse": [
            HookMatcher(matcher="mcp__soporte__cerrar_incidencia", hooks=[proteger_cierres]),
            HookMatcher(hooks=[auditar]),
        ],
        "PostToolUse": [HookMatcher(hooks=[auditar])],
    },
)

`INC-013` (Hooli, webhook de pedidos) es de prioridad alta. `cerrar_incidencia` está en `allowed_tools`: aun así, el hook la bloquea.

In [ ]:
async for msg in query(prompt="Cierra la incidencia INC-013 de Hooli, ya está resuelta.", options=options):
    imprimir(msg)

In [ ]:
herramientas.obtener("INC-013")["estado"]

Se deniega y el modelo explica el motivo.

| | `can_use_tool` (3.2) | hook `PreToolUse` |
|---|---|---|
| Cuándo se ejecuta | solo si el permiso **no** está ya resuelto (no está en `allowed_tools`…) | **siempre**, para cada llamada |
| Para qué | decidir lo que queda sin decidir (p. ej. preguntar a una persona) | aplicar **políticas** y observar |

Para políticas, hooks. Otros eventos útiles: `UserPromptSubmit`, `Stop`, `SubagentStart` / `SubagentStop`, `PreCompact`.

## Persistencia de sesiones

El SDK guarda cada conversación en disco con un `session_id`. Ya guardamos el de la primera ejecución en `logs/ultima_sesion.txt`:

In [ ]:
Path("../../logs/ultima_sesion.txt").read_text()

**Ahora reinicia el kernel** (botón **Restart** de la barra del notebook). El panel Variables se queda vacío: no queda nada en la memoria de Python.

Después ejecuta **solo** las dos celdas siguientes.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import herramientas
from taller_utils import (
    imprimir, buscar_incidencias, crear_incidencia, cerrar_incidencia,
    CLASIFICADOR, HERRAMIENTAS_KB,
)
from claude_agent_sdk import query, create_sdk_mcp_server, ClaudeAgentOptions, HookMatcher, ResultMessage

soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[buscar_incidencias, crear_incidencia, cerrar_incidencia])
SYSTEM = ("Eres un agente de soporte de nivel 1. Respondes en español y de forma breve. "
          "Antes de crear una incidencia, comprueba que no existe ya una abierta sobre el mismo problema. "
          "La prioridad de cada incidencia se asigna según los criterios oficiales de prioridad: no la supongas.")
BASE = dict(
    model=MODEL,
    system_prompt=SYSTEM,
    mcp_servers={"soporte": soporte},
    tools=["Task"],
    allowed_tools=["Task", "mcp__soporte__buscar_incidencias", "mcp__soporte__crear_incidencia",
                   "mcp__soporte__cerrar_incidencia", *HERRAMIENTAS_KB],
    agents={"clasificador": CLASIFICADOR},   # con su propio servidor de la KB
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},
    max_turns=15,
    setting_sources=[],
)

In [ ]:
session_id_guardado = Path("../../logs/ultima_sesion.txt").read_text().strip()
options = ClaudeAgentOptions(**BASE, resume=session_id_guardado)

async for msg in query(prompt="¿Qué incidencia acabamos de crear para Initech?", options=options):
    imprimir(msg)

Lo sabe. **La API sigue sin memoria:** es el SDK quien ha guardado la conversación y la reenvía.

### Abrir una rama: `fork_session`

Con `fork_session=True` junto a `resume`, la conversación continúa en una sesión **nueva** a partir de ese punto, sin tocar la original. Útil para probar alternativas.

In [ ]:
options = ClaudeAgentOptions(**BASE, resume=session_id_guardado, fork_session=True)

async for msg in query(prompt="Si en vez de todo el equipo fuera una sola persona, ¿qué prioridad le habrías puesto?",
                       options=options):
    imprimir(msg)
    if isinstance(msg, ResultMessage):
        print("sesión original:", session_id_guardado)
        print("sesión nueva:   ", msg.session_id)

In [ ]:
herramientas.restaurar_datos()

## Resumen del bloque

| Problema | Capacidad |
|---|---|
| herramientas propias bien definidas | `@tool` y servidor in-process |
| reutilizar integraciones | MCP externo |
| aislar contexto y especializar | subagentes |
| observar y aplicar políticas | hooks |
| continuidad | sesiones (`resume`, `fork_session`) |

`auditar`, `proteger_cierres` y `guardar_sesion` quedan en `taller_utils.py`.

**En el siguiente bloque** juntamos todo en un proyecto real y lo rompemos a propósito.